# 2. Deep Agents: a richer agent harness

Notebook 1 assembled the pieces ourselves.

This notebook keeps the same **Manager, Analyst and Reviewer** idea but uses **Deep Agents**.

We demonstrate native subagents, built-in file tools, shell execution, optional task planning and parallel delegation.

## 1. Imports and setup

**Important:** `LocalShellBackend` is convenient for a classroom demo but runs shell commands on the local machine. It is not a security sandbox.

In [1]:
from pathlib import Path
import re

import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv

from deepagents import create_deep_agent
from deepagents.backends import LocalShellBackend
from langchain.agents.middleware import TodoListMiddleware
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

load_dotenv()

MODEL = "gpt-5.6-luna"
model = ChatOpenAI(model=MODEL, use_responses_api=True)
search_model = ChatOpenAI(model=MODEL, use_responses_api=True)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

## 2. Recreate the small local knowledge base

In [2]:
documents = [
    Document(
        page_content=(
            "Motor claims inflation rose because repair labour, replacement parts, "
            "vehicle technology and hire-car costs became more expensive. "
            "The insurer responded by increasing pricing and tightening claims controls."
        ),
        metadata={"source": "motor_claims_note.txt"},
    ),
    Document(
        page_content=(
            "A household insurer is testing generative AI to summarise claim notes. "
            "The pilot is intended to reduce administrative work, but human claims handlers "
            "remain responsible for coverage decisions and settlement authority."
        ),
        metadata={"source": "claims_ai_pilot.txt"},
    ),
    Document(
        page_content=(
            "Fraud teams combine rules, anomaly detection and investigator judgement. "
            "An AI assistant may help investigators search past cases, but false positives "
            "can create unnecessary referrals and customer friction."
        ),
        metadata={"source": "fraud_note.txt"},
    ),
    Document(
        page_content=(
            "Personally identifiable information in insurance can include customer names, "
            "email addresses, policy numbers, payment details and claim identifiers. "
            "Sensitive information should be minimised before it is sent to external systems."
        ),
        metadata={"source": "privacy_note.txt"},
    ),
]

vector_store = InMemoryVectorStore(embedding=embeddings)
vector_store.add_documents(documents)

['990a4ae6-1930-4ff6-b9b9-465bb66fd4ac',
 'd13188e9-a8c3-455a-8377-962e2d8d822a',
 '58719fbd-ed49-42a4-89e0-678293b7c3d4',
 '998bee80-b12a-46cd-8a3f-cd5274d6cb55']

## 3. Research tools

Deep Agents accepts ordinary LangChain tools. We keep web search, fetch, semantic retrieval and keyword retrieval as custom tools.

File and shell tools are not recreated because the Deep Agents backend supplies them.

In [3]:
@tool
def web_search(query: str) -> str:
    """Search the live web using OpenAI's built-in web-search tool."""
    response = search_model.invoke(query, tools=[{"type": "web_search"}])

    lines = []
    for block in response.content_blocks:
        if block.get("type") == "text":
            lines.append(block.get("text", ""))
            for citation in block.get("annotations", []):
                if citation.get("url"):
                    lines.append(f"Source: {citation['url']}")
    return "\n".join(lines)


@tool
def fetch_url(url: str) -> str:
    """Fetch one web page and return readable text from it."""
    html = requests.get(url, timeout=20).text
    text = BeautifulSoup(html, "html.parser").get_text(" ", strip=True)
    return text[:12000]


@tool
def semantic_search(query: str) -> str:
    """Search the local teaching notes by meaning."""
    matches = vector_store.similarity_search(query, k=3)
    return "\n\n".join(
        f"{doc.metadata['source']}: {doc.page_content}"
        for doc in matches
    )


@tool
def keyword_search(query: str) -> str:
    """Search the local teaching notes using exact query words."""
    words = set(re.findall(r"\w+", query.lower()))
    scored = []

    for doc in documents:
        text_words = set(re.findall(r"\w+", doc.page_content.lower()))
        scored.append((len(words & text_words), doc))

    scored.sort(key=lambda item: item[0], reverse=True)

    return "\n\n".join(
        f"score={score} | {doc.metadata['source']}: {doc.page_content}"
        for score, doc in scored[:3]
    )

### Wrapping an existing function as a tool

`deep_agent_workspace/prices.py` already has a plain Python function, `get_price_series_yahoo`, that fetches a daily price history from Yahoo Finance via the `yfinance` package. Wrapping it with `@tool` is enough to make it callable by the agent -- we don't need to rewrite the underlying logic, just give the model a clear docstring and return a string it can read.

In [ ]:
from deep_agent_workspace.prices import get_price_series_yahoo


@tool
def yahoo_price_series(ticker: str, period: str = "1y") -> str:
    """Look up a daily OHLCV price history for a stock ticker from Yahoo Finance.

    `period` is one of "1mo", "3mo", "6mo", "1y", "2y", "5y", "10y", "ytd", "max".
    Returns the oldest and most recent rows plus the row count, not the full series,
    to keep the tool's reply short enough for the agent to read easily.
    """
    rows = get_price_series_yahoo(ticker, period=period)
    first, last = rows[0], rows[-1]
    return (
        f"{ticker.upper()}: {len(rows)} trading days from {first['date']} to {last['date']}.\n"
        f"First close: {first['close']}\n"
        f"Last close: {last['close']}"
    )

## 4. Give Deep Agents a workspace

With this backend, Deep Agents can expose file operations such as `ls`, `read_file`, `write_file`, `edit_file`, `delete`, `glob` and `grep`, plus `execute` for shell commands.

In [4]:
workspace = Path("deep_agent_workspace")
workspace.mkdir(exist_ok=True)

backend = LocalShellBackend(
    root_dir=str(workspace),
    virtual_mode=True,
)

## 5. Define specialist subagents

If `tools` is omitted, a custom subagent inherits the parent's custom tools.

The Manager delegates using Deep Agents' built-in `task` tool.

In [5]:
subagents = [
    {
        "name": "analyst",
        "description": (
            "Researches insurance and actuarial questions using web and local evidence. "
            "Use for fact finding, comparisons and numerical analysis."
        ),
        "system_prompt": (
            "You are the Analyst. Research carefully. Use web_search for current evidence, "
            "semantic_search for meaning-based local retrieval, keyword_search for exact terms, "
            "and fetch_url when you need one specific page. "
            "Return a concise evidence-based report with source URLs when available."
        ),
    },
    {
        "name": "reviewer",
        "description": (
            "Independently reviews an analysis for unsupported claims, missing risks, "
            "weak evidence and numerical inconsistencies."
        ),
        "system_prompt": (
            "You are the Reviewer. Challenge the work rather than merely summarising it. "
            "Use research tools when independent checking is useful. "
            "Return PASS or REVISE, followed by specific reasons."
        ),
    },
]

## 6. Create the Deep Agent Manager

`TodoListMiddleware` adds the optional `write_todos` planning tool.

In [ ]:
deep_manager = create_deep_agent(
    model=model,
    tools=[web_search, fetch_url, semantic_search, keyword_search, yahoo_price_series],
    subagents=subagents,
    backend=backend,
    middleware=[TodoListMiddleware()],
    system_prompt=(
        "You are the Manager. Use the analyst subagent for substantive research and "
        "the reviewer subagent to challenge important conclusions. "
        "For a multi-step task, use write_todos first. "
        "Use the workspace to save useful intermediate notes. "
        "For substantial answers, do not skip independent review."
    ),
)

## 7. First Deep Agents run

In [7]:
question = (
    "Prepare a short briefing on realistic generative-AI uses in insurance claims. "
    "Include benefits, failure modes and controls. Use current web evidence where useful. "
    "Save the final briefing as /final_briefing.md in the workspace."
)

result = deep_manager.invoke({
    "messages": [{"role": "user", "content": question}]
})

print(result["messages"][-1].text)

Created the final briefing at:

`/final_briefing.md`

It includes realistic claims use cases, expected benefits, failure modes, practical controls, current regulatory evidence, and a 90-day implementation plan.


## 8. Use the built-in filesystem tools

In [8]:
result = deep_manager.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "List the files in the workspace, then use grep to find occurrences "
            "of the word 'risk'. Briefly report what you found."
        ),
    }]
})

print(result["messages"][-1].text)

Files in the workspace:

- `/draft_briefing.md`
- `/final_briefing.md`

The word **“risk”** appears in both files, with matching occurrences. It is used in contexts including:

- Bias and proxy-discrimination risks
- Risk-tiering AI use cases
- Supplier, incident, and governance risks
- Risk-based regulatory oversight
- Recommendations to inventory and risk-tier claims AI experiments
- Regulatory notes concerning high-risk AI systems and data-protection safeguards


## 9. Demonstrate shell execution

Keep the classroom example harmless.

In [9]:
result = deep_manager.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Use the execute tool to run 'python --version' and 'pwd'. "
            "Tell me what they show."
        ),
    }]
})

print(result["messages"][-1].text)

- `python --version`: `python` is not installed or not on the PATH (`python: not found`).
- `pwd`: `/home/nodozi/projects/Intern_GenAI_Examples/deep_agent_workspace`


## 10. Parallel subagents

A Deep Agent can issue several `task` calls in one turn, allowing delegated work to run in parallel.

In [10]:
result = deep_manager.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Investigate these two questions independently and in parallel where possible: "
            "(1) benefits of generative AI in claims operations, "
            "(2) privacy and governance risks. "
            "Use the analyst for the first and the reviewer as an independent risk researcher "
            "for the second, then combine the findings."
        ),
    }]
})

print(result["messages"][-1].text)

OpenAIRateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-5.6-luna in organization org-ZY61ZSJV3PB32Ulw0zXVC6jp on tokens per min (TPM): Limit 200000, Used 194389, Requested 28665. Please try again in 6.916s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}

## 11. Inspect the run

In [ ]:
for message in result["messages"]:
    message.pretty_print()

## 12. Try the price-lookup tool

In [ ]:
result = deep_manager.invoke({
    "messages": [{
        "role": "user",
        "content": "What has Legal & General's (ticker LGEN.L) share price done over the last year?",
    }]
})

print(result["messages"][-1].text)

## Why this feels different from Notebook 1

**Notebook 1:** we created file tools, shell execution and agent-as-tool delegation ourselves.

**Notebook 2:** Deep Agents supplies a harness around the same core LangChain ideas.

### Common gotchas

- Deep Agents subagents have fresh context; they do not continuously chat with one another.
- A subagent returns a handoff to the parent.
- `LocalShellBackend` is convenient, but it is not isolated from your computer.
- More tools are not always better.
- Planning is opt-in, so we explicitly add `TodoListMiddleware`.